# EDA cleaning and flagging

In this notebook we check the two raw TMDB files and make a flagged copy for analysis.

We do not automatically remove rows with missing or unusual values. First, we check whether the value is actually incorrect or whether it is expected because of the person's role. In our dataset, actors and crew members have different attributes, so some fields are not applicable to certain roles. We keep these records and add flags to document the reason for the missing or unusual values.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

root = Path.cwd()
raw_dir = root / 'data' / 'raw'
clean_dir = root / 'data' / 'clean'

if not (raw_dir / 'movies.parquet').exists():
    raise FileNotFoundError(
        f'Raw data not found in {raw_dir}. Run this notebook from the project root.'
    )

movies = pd.read_parquet(raw_dir / 'movies.parquet')
credits = pd.read_parquet(raw_dir / 'credits.parquet')

print('movies shape:', movies.shape)
print('credits shape:', credits.shape)


movies shape: (1208, 9)
credits shape: (14252, 7)


In [2]:
# Count actors and identify short cast lists.
actor_counts = credits[credits['role'].str.lower() == 'actor'].groupby('movie_id').size().rename('actor_count')

movies_flagged = movies.merge(actor_counts.rename('actor_count'), left_on='id', right_index=True, how='left')
movies_flagged['cast_short_flag'] = movies_flagged['actor_count'].lt(6).fillna(False)

movies_flagged[['id', 'title', 'actor_count', 'cast_short_flag']].head()


,id,title,actor_count,cast_short_flag
0,755898,War of the Worlds,6.0,False
1,591634,False Positive,6.0,False
2,872585,Oppenheimer,6.0,False
3,361743,Top Gun: Maverick,6.0,False
4,1368166,The Housemaid,6.0,False


## Basic data checks

These checks reproduce the main results from `findings.md`: duplicates, missing values, and links between the two tables.


In [3]:
print('Movie dtypes:')
display(movies.dtypes)

print('Credit dtypes:')
display(credits.dtypes)

print('Duplicate movie IDs:', movies['id'].duplicated().sum())
print('Duplicate credit rows:', credits.duplicated().sum())
print('Unique movie IDs:', movies['id'].nunique())
print('Credits without a selected movie:', (~credits['movie_id'].isin(movies['id'])).sum())

print('Missing values in movies:')
display(movies.isna().sum())

print('Missing values in credits:')
display(credits.isna().sum())


Movie dtypes:


id                     int64
title                 object
original_title        object
release_date          object
genre_ids             object
original_language     object
vote_count             int64
vote_average         float64
popularity           float64
dtype: object

Credit dtypes:


movie_id           int64
person_id          int64
person_name       object
role              object
department        object
job               object
billing_order    float64
dtype: object

Duplicate movie IDs: 0
Duplicate credit rows: 0
Unique movie IDs: 1208
Credits without a selected movie: 0
Missing values in movies:


id                   0
title                0
original_title       0
release_date         0
genre_ids            0
original_language    0
vote_count           0
vote_average         0
popularity           0
dtype: int64

Missing values in credits:


movie_id            0
person_id           0
person_name         0
role                0
department       7191
job              7191
billing_order    7061
dtype: int64

## Missingness and valid ranges

The cross-tabs check whether missing values are explained by the person's role. The assertions check the basic numeric limits from the API fields.


In [4]:
print('Missing department by role:')
display(pd.crosstab(credits['role'], credits['department'].isna()))

print('Missing job by role:')
display(pd.crosstab(credits['role'], credits['job'].isna()))

print('Missing billing order by role:')
display(pd.crosstab(credits['role'], credits['billing_order'].isna()))

assert movies['vote_count'].ge(100).all()
assert movies['vote_average'].between(0, 10).all()
assert movies['popularity'].ge(0).all()
print('Numeric range checks: OK')

display(movies[['vote_count', 'vote_average', 'popularity']].describe())


Missing department by role:


department,False,True
role,,
actor,0,7191
cinematographer,1199,0
director,1289,0
producer,4573,0


Missing job by role:


job,False,True
role,,
actor,0,7191
cinematographer,1199,0
director,1289,0
producer,4573,0


Missing billing order by role:


billing_order,False,True
role,,
actor,7191,0
cinematographer,0,1199
director,0,1289
producer,0,4573


Numeric range checks: OK


,vote_count,vote_average,popularity
count,1208.000000,1208.000000,1208.000000
mean,936.350993,6.454498,7.869145
std,1446.194988,0.723974,7.157796
min,100.000000,4.034000,1.165100
25%,186.000000,5.973000,3.988500
50%,378.000000,6.438500,5.626500
75%,1046.000000,6.948000,9.251075
max,12616.000000,8.400000,107.044500


In [5]:
credits_flagged = credits.copy()

# Explain missing values using the person's role.
credits_flagged['missing_reason'] = np.where(
    credits_flagged['role'].str.lower() == 'actor',
    'not_applicable_for_actor',
    np.where(credits_flagged['role'].str.lower().isin(['director', 'producer', 'cinematographer']), 'not_applicable_for_crew', 'unknown')
)
# Add the short-cast flag to the credits table too.
actor_counts = credits_flagged[credits_flagged['role'].str.lower() == 'actor'].groupby('movie_id').size().rename('actor_count')
credits_flagged = credits_flagged.merge(actor_counts.rename('actor_count'), on='movie_id', how='left')
credits_flagged['cast_short_flag'] = credits_flagged['actor_count'].lt(6).fillna(False)
credits_flagged[['movie_id', 'person_name', 'role', 'department', 'job', 'billing_order', 'missing_reason', 'cast_short_flag']].head(10)


,movie_id,person_name,role,department,job,billing_order,missing_reason,cast_short_flag
0,755898,Rich Lee,director,Directing,Director,NaN,not_applicable_for_crew,False
1,755898,Timur Bekmambetov,producer,Production,Producer,NaN,not_applicable_for_crew,False
2,755898,Patrick Aiello,producer,Production,Producer,NaN,not_applicable_for_crew,False
3,755898,Christopher Probst,cinematographer,Camera,Director of Photography,NaN,not_applicable_for_crew,False
4,755898,Ice Cube,actor,None,None,0.0,not_applicable_for_actor,False
5,755898,Eva Longoria,actor,None,None,1.0,not_applicable_for_actor,False
6,755898,Clark Gregg,actor,None,None,2.0,not_applicable_for_actor,False
7,755898,Iman Benson,actor,None,None,3.0,not_applicable_for_actor,False
8,755898,Henry Hunter Hall,actor,None,None,4.0,not_applicable_for_actor,False
9,755898,Devon Bostick,actor,None,None,5.0,not_applicable_for_actor,False


## Cleaning decisions

Our main decision is to keep the records and add information about their quality.

- We keep all rows from the raw dataset.
- Missing values caused by the person's role are described with `missing_reason`.
- We do not replace these values with zero, a median, or made-up text.
- Movies with fewer than six actors are kept and marked with `cast_short_flag`.
- The project scope is documented in this notebook and in the data card; it describes the sample and is not a row-level data error.
- High popularity or vote counts were checked and kept because they are possible values; no extra flag is needed.


In [6]:
# Save flagged copies without overwriting the raw files.
clean_dir.mkdir(parents=True, exist_ok=True)
movies_flagged.to_parquet(clean_dir / 'movies_flagged.parquet', index=False)
credits_flagged.to_parquet(clean_dir / 'credits_flagged.parquet', index=False)
print('Saved:', clean_dir / 'movies_flagged.parquet')
print('Saved:', clean_dir / 'credits_flagged.parquet')


Saved: D:\Master\Data Acquisition\Project\tmdb-data-acquisition\data\clean\movies_flagged.parquet
Saved: D:\Master\Data Acquisition\Project\tmdb-data-acquisition\data\clean\credits_flagged.parquet


## Leakage note

No predictive model or target variable is used in this milestone, so train/test leakage is not applicable. This notebook focuses on data quality and traceable cleaning decisions.


## Final note

The result of this notebook is two flagged Parquet files in `data/clean/`. The raw data stays unchanged. Our approach is: find an unusual case, explain it, add a useful flag when needed, and keep the row unless it is really invalid. For this project, we did not find a reason to delete any rows.
